# IRRM-CODEC inverse: run and analyze

This notebook mirrors the forward example, but trains and evaluates the inverse model that reconstructs a CDR3 sequence from a TCRemP embedding.

## 1. Define inputs

The AIRR file must contain `clone_id`, `junction_aa`, `v_call`, `j_call`, `locus`.
The embeddings parquet must contain `clone_id` plus either `tcremp_emb` or numeric embedding columns.

In [ ]:
from pathlib import Path

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
airr_path = '/projects/immunestatus/vdjdb_olga/airr_format/trb_background_100k.tsv'  # root / 'data' / 'sample_airr.tsv'
embeddings_path = '/projects/immunestatus/vdjdb_olga/tcremp/trb_background_100k_embeddings.parquet'
output_dir = root / 'artifacts' / 'inverse_demo_trb'

airr_path, embeddings_path, output_dir

In [ ]:
import sys

sys.path.insert(0, str(root))

## 2. Preview the two files

In [ ]:
import pandas as pd

airr_df = pd.read_csv(airr_path, sep='\t')
emb_df = pd.read_parquet(embeddings_path)

display(airr_df.head())
display(emb_df.head())

## 3. Launch training

In [ ]:
import subprocess

cmd = [
    sys.executable, '-m', 'irrm_codec.train_inverse',
    '--airr-path', str(airr_path),
    '--embeddings-path', str(embeddings_path),
    '--output-dir', str(output_dir),
    '--locus', 'beta',
    '--epochs', '25',
]
subprocess.run(cmd, cwd=root, check=True)
cmd

## 4. Inspect saved metrics and merge stats

In [ ]:
import json
import pandas as pd

history = json.loads((output_dir / 'history.json').read_text())
test_metrics = json.loads((output_dir / 'test_metrics.json').read_text())
data_stats = json.loads((output_dir / 'data_stats.json').read_text())
history_df = pd.json_normalize(history)

display(history_df)
display(test_metrics)
display(data_stats)

In [ ]:
history_df.plot(x='epoch', y=['train.loss', 'val.loss'], figsize=(8, 4), grid=True, title='Loss by epoch')

In [ ]:
history_df.plot(
    x='epoch',
    y=['val.token_accuracy', 'val.length_accuracy', 'val.exact_match'],
    figsize=(8, 4),
    grid=True,
    title='Validation metrics by epoch',
)

## 5. Load a checkpoint

In [ ]:
import torch
from irrm_codec.inverse_model import InverseModel

checkpoint = torch.load(output_dir / 'best.pt', map_location='cpu')
extra = checkpoint['extra']
model = InverseModel(
    embedding_dim=extra['embedding_dim'],
    max_len=extra.get('max_len', 40),
)
model.load_state_dict(checkpoint['model_state'])
model.eval()
checkpoint['metrics']

## 6. Test the model on held-out examples

In [ ]:
import numpy as np
from irrm_codec.dataio import load_airr_with_embeddings
from irrm_codec.tokenization import decode
from irrm_codec.utils import apply_standardizer, split_indices

df, emb, _merge_stats = load_airr_with_embeddings(
    airr_path=airr_path,
    embeddings_path=embeddings_path,
    locus='beta',
)

train_idx, val_idx, test_idx = split_indices(len(df), train_fraction=0.8, val_fraction=0.1, seed=42)
test_df = df.iloc[test_idx].reset_index(drop=True)
test_emb = apply_standardizer(
    emb[test_idx],
    np.load(output_dir / 'mean.npy'),
    np.load(output_dir / 'std.npy'),
)

sample_size = 10
sample_emb = torch.tensor(test_emb[:sample_size], dtype=torch.float32)
pred_tokens, pred_lengths = model.generate(sample_emb, max_len=model.max_len)

results = pd.DataFrame({
    'target': test_df['junction_aa'].iloc[:sample_size].tolist(),
    'predicted': [decode(row.tolist()) for row in pred_tokens],
    'predicted_length': pred_lengths.tolist(),
})
results['target_length'] = results['target'].str.len()
results['exact_match'] = results['target'] == results['predicted']
results

In [ ]:
results['exact_match'].mean()